# Detection con ELIta
Questo notebook applica il lessico **ELIta** (originale e ricalcolato) ai commenti raccolti da r/Italia.

**Input**:
- `corpus_italy_film.csv`: commenti con metadati
- `tokens_italy_film.csv`: token con POS-tag
- `ELIta_INTENSITY_Matrix.csv`: matrice ELIta originale (Fase 1)
- Matrici ricalcolate con α = 0.2 / 0.5 / 0.8 (da `Valutazione_e_Ricalcolo.ipynb`)

**Output**:
- Tabella emozione / n.commenti / n.aggettivi (richiesta dalle specifiche)
- Confronto tra ELIta originale e versioni ricalcolate
- Grafici comparativi

## Import e caricamento dati

In [1]:
import pandas as pd
import numpy as np
import emoji
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.metrics.pairwise import cosine_similarity
from pathlib import Path

# --- Percorsi file ---
CORPUS_CSV   = "corpus_Italia_film.csv"
TOKENS_CSV   = "tokens_Italia_film.csv"
ELITA_CSV    = "../Fase1/ELIta_INTENSITY_Matrix.csv"
ALPHA_02_CSV = "../Fase2/output_csv/elita_recalculated_0_2.csv"
ALPHA_05_CSV = "../Fase2/output_csv/elita_recalculated_0_5.csv"
ALPHA_08_CSV = "../Fase2/output_csv/elita_recalculated_0_8.csv"
OUTPUT_DIR   = Path("output_confronto")
OUTPUT_DIR.mkdir(exist_ok=True)

from Fase1.emotion_config import BASIC_EMOTIONS, EMOTION_COLORS

# Caricamento corpus e token
df_corpus = pd.read_csv(CORPUS_CSV)
df_tokens = pd.read_csv(TOKENS_CSV)

print(f"Commenti              : {len(df_corpus)}")
print(f"Token totali          : {len(df_tokens)}")

Commenti              : 746
Token totali          : 62282


## Caricamento e preparazione matrici ELIta

In [2]:
# Matrice originale — solo parole (no emoji)
df_matrix = pd.read_csv(ELITA_CSV, index_col=0)

def is_not_emoji(text):
    return emoji.emoji_count(str(text)) == 0

df_elita_orig = df_matrix[df_matrix.index.map(is_not_emoji)][BASIC_EMOTIONS].fillna(0)
print(f"ELIta originale: {len(df_elita_orig)} parole")
df_elita_orig.head(5)

ELIta originale: 6719 parole


,gioia,tristezza,rabbia,paura,disgusto,fiducia,sorpresa,aspettativa
parola,,,,,,,,
????,0.12,0.42,0.42,0.17,0.17,0.04,0.79,0.17
a_caso,0.04,0.21,0.29,0.54,0.08,0.04,0.83,0.21
a_malincuore,0.00,0.83,0.21,0.21,0.21,0.00,0.04,0.12
a_scanso_di,0.17,0.25,0.42,0.17,0.04,0.25,0.21,0.58
abbagliante,0.46,0.04,0.12,0.12,0.00,0.17,0.71,0.62


In [3]:
df_elita_02 = pd.read_csv(ALPHA_02_CSV, index_col=0)
df_elita_05 = pd.read_csv(ALPHA_05_CSV, index_col=0)
df_elita_08 = pd.read_csv(ALPHA_08_CSV, index_col=0)

MATRICES = {
    "Originale (α=0)" : df_elita_orig,
    "Ibrido (α=0.2)"  : df_elita_02[BASIC_EMOTIONS].fillna(0),
    "Ibrido (α=0.5)"  : df_elita_05[BASIC_EMOTIONS].fillna(0),
    "Ibrido (α=0.8)"  : df_elita_08[BASIC_EMOTIONS].fillna(0),
}
print("\nMatrici pronte:", list(MATRICES.keys()))
MATRICES


Matrici pronte: ['Originale (α=0)', 'Ibrido (α=0.2)', 'Ibrido (α=0.5)', 'Ibrido (α=0.8)']


{'Originale (α=0)':               gioia  tristezza  rabbia  paura  disgusto  fiducia  sorpresa  \
 parola                                                                       
 ????           0.12       0.42    0.42   0.17      0.17     0.04      0.79   
 a_caso         0.04       0.21    0.29   0.54      0.08     0.04      0.83   
 a_malincuore   0.00       0.83    0.21   0.21      0.21     0.00      0.04   
 a_scanso_di    0.17       0.25    0.42   0.17      0.04     0.25      0.21   
 abbagliante    0.46       0.04    0.12   0.12      0.00     0.17      0.71   
 ...             ...        ...     ...    ...       ...      ...       ...   
 zona           0.08       0.00    0.00   0.04      0.04     0.04      0.00   
 zotico         0.00       0.21    0.42   0.42      0.50     0.04      0.04   
 zucca          0.29       0.00    0.00   0.00      0.25     0.04      0.00   
 zucchero       0.71       0.17    0.00   0.04      0.04     0.29      0.04   
 zucchina       0.12       0.12  

## Funzione di emotion detection
Per ogni commento:
1. Prendiamo i **lemmi più significativi** (POS = ADJ, NOUN, VERB) trovati da spaCy
2. Li cerchiamo nella matrice ELIta
3. Per ogni aggettivo trovato, prendiamo il vettore di intensità emotiva
4. Sommiamo i vettori → profilo emotivo del commento

In [4]:
def detect_emotions(df_corpus, df_tokens, df_elita):
    """
    Applica ELIta ai token di ogni commento (ADJ + NOUN + VERB).

    Restituisce:
        df_results : DataFrame con profilo emotivo per commento
        df_table   : tabella emozione / N.Commenti / N.Token
        unmatched  : lemmi non trovati in ELIta
    """
    # Filtra per ADJ, NOUN, VERB
    pos_filter = {"ADJ", "NOUN", "VERB"}
    df_filtered = df_tokens[df_tokens["pos"].isin(pos_filter)].copy()

    elita_index = set(df_elita.index)

    tokens_by_comment = df_filtered.groupby("comment_id")["lemma"].apply(list).to_dict()

    results = []
    unmatched_global = []

    for _, row in df_corpus.iterrows():
        cid   = row["comment_id"]
        lemmi = tokens_by_comment.get(cid, [])

        emotion_scores = {emo: 0.0 for emo in BASIC_EMOTIONS}
        found, not_found = [], []

        for lemma in lemmi:
            if lemma in elita_index:
                found.append(lemma)
                for emo in BASIC_EMOTIONS:
                    emotion_scores[emo] += df_elita.loc[lemma, emo]
            else:
                not_found.append(lemma)

        unmatched_global.extend(not_found)

        results.append({
            "comment_id"      : cid,
            "n_tokens_total"  : len(lemmi),
            "n_tokens_matched": len(found),
            "tokens_matched"  : found,
            **emotion_scores,
            "dominant_emotion": max(emotion_scores, key=emotion_scores.get)
                                 if any(v > 0 for v in emotion_scores.values()) else "neutrale"
        })

    df_results = pd.DataFrame(results)

    # Tabella riassuntiva richiesta dalle specifiche
    table_rows = []
    for emo in BASIC_EMOTIONS:
        comm_with_emo = df_results[df_results[emo] > 0]["comment_id"]
        n_tokens_emo  = df_filtered[
            df_filtered["comment_id"].isin(comm_with_emo) &
            df_filtered["lemma"].isin(elita_index)
        ]["lemma"].count()
        table_rows.append({
            "Emozione"  : emo.capitalize(),
            "N. Commenti": int((df_results[emo] > 0).sum()),
            "N. Token"   : int(n_tokens_emo)
        })
    df_table = pd.DataFrame(table_rows)

    return df_results, df_table, list(set(unmatched_global))


print("Funzione definita.")

Funzione definita.


## Applicazione a tutte le versioni di ELIta

In [5]:
all_results = {}
all_tables  = {}

for version_name, df_elita in MATRICES.items():
    print(f"\nProcesso: {version_name}")
    df_res, df_tab, unmatched = detect_emotions(df_corpus, df_tokens, df_elita)
    all_results[version_name] = df_res
    all_tables[version_name]  = df_tab

    total   = df_res["n_tokens_total"].sum()
    matched = df_res["n_tokens_matched"].sum()
    rate    = matched / total * 100 if total > 0 else 0
    print(f"  Token trovati in ELIta       : {matched}/{total} ({rate:.1f}%)")
    print(f"  Commenti con almeno 1 match  : {(df_res['n_tokens_matched'] > 0).sum()}")

print("\nDone.")


Processo: Originale (α=0)
  Token trovati in ELIta       : 18714/25028 (74.8%)
  Commenti con almeno 1 match  : 744

Processo: Ibrido (α=0.2)
  Token trovati in ELIta       : 18714/25028 (74.8%)
  Commenti con almeno 1 match  : 744

Processo: Ibrido (α=0.5)
  Token trovati in ELIta       : 18714/25028 (74.8%)
  Commenti con almeno 1 match  : 744

Processo: Ibrido (α=0.8)
  Token trovati in ELIta       : 18714/25028 (74.8%)
  Commenti con almeno 1 match  : 744

Done.


## Tabella riassuntiva

In [6]:
# Tabella per ELIta originale — quella richiesta dalle specifiche
print("=" * 50)
print("TABELLA EMOTION DETECTION — ELIta Originale")
print("=" * 50)
display(all_tables["Originale (α=0)"])

# Salviamo il CSV
all_tables["Originale (α=0)"].to_csv(OUTPUT_DIR / "emotion_table_originale.csv", index=False)
print("\nSalvata in 'output_confronto/emotion_table_originale.csv'")

TABELLA EMOTION DETECTION — ELIta Originale


,Emozione,N. Commenti,N. Token
0,Gioia,744,18714
1,Tristezza,744,18714
2,Rabbia,744,18714
3,Paura,744,18714
4,Disgusto,744,18714
5,Fiducia,744,18714
6,Sorpresa,744,18714
7,Aspettativa,744,18714



Salvata in 'output_confronto/emotion_table_originale.csv'


## Confronto tra versioni ELIta
Qualitativamente, vogliamo capire:
- Come cambia la distribuzione dell'emozione dominante tra le versioni?
- Come cambiano i punteggi medi per ogni emozione?

In [7]:
# Grafico: distribuzione emozione dominante per ogni versione
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=list(MATRICES.keys()),
    vertical_spacing=0.15
)

positions = [(1,1), (1,2), (2,1), (2,2)]

for idx, (version_name, df_res) in enumerate(all_results.items()):
    row, col = positions[idx]
    counts = df_res["dominant_emotion"].value_counts()
    
    for emo in BASIC_EMOTIONS:
        n = counts.get(emo, 0)
        fig.add_trace(
            go.Bar(
                name=emo,
                x=[emo],
                y=[n],
                marker_color=EMOTION_COLORS[emo],
                showlegend=(idx == 0),
                legendgroup=emo
            ),
            row=row, col=col
        )

fig.update_layout(
    title_text="Distribuzione emozione dominante per versione ELIta",
    barmode="group",
    height=700
)
fig.show()

Vediamo che gioia e aspettativa tendono sempre a dominare.

---

In [8]:
# Grafico: confronto punteggi medi per emozione tra le 4 versioni
means = {}
for version_name, df_res in all_results.items():
    means[version_name] = df_res[BASIC_EMOTIONS].mean()

df_means = pd.DataFrame(means).T.reset_index().rename(columns={"index": "Versione"})
df_melted = df_means.melt(id_vars="Versione", var_name="Emozione", value_name="Punteggio medio")

fig2 = px.bar(
    df_melted,
    x="Emozione", y="Punteggio medio",
    color="Versione",
    barmode="group",
    title="Punteggio emotivo medio per versione ELIta",
    color_discrete_sequence=["#455A64", "#1E88E5", "#FB8C00", "#E53935"]
)
fig2.show()

Questo grafico mostra come i punteggi medi per ogni emozione cambiano tra le versioni. In generale, l'ibrido con α=0.5 sembra bilanciare meglio le emozioni, mentre l'originale tende a dare punteggi più bassi

---

In [9]:
# Tabella confronto: quanti commenti cambiano emozione dominante tra originale e α=0.5?
orig  = all_results["Originale (α=0)"][["comment_id", "dominant_emotion"]].rename(columns={"dominant_emotion": "orig"})
alpha05 = all_results["Ibrido (α=0.5)"][["comment_id", "dominant_emotion"]].rename(columns={"dominant_emotion": "alpha05"})

df_compare = orig.merge(alpha05, on="comment_id")
cambiati = (df_compare["orig"] != df_compare["alpha05"]).sum()
totale   = len(df_compare)

print(f"Commenti che cambiano emozione dominante (Originale → α=0.5): {cambiati}/{totale} ({cambiati/totale*100:.1f}%)")
print()
print("Matrice di transizione (Originale → Ibrido α=0.5):")
pd.crosstab(df_compare["orig"], df_compare["alpha05"], margins=True)

Commenti che cambiano emozione dominante (Originale → α=0.5): 79/746 (10.6%)

Matrice di transizione (Originale → Ibrido α=0.5):


alpha05,aspettativa,fiducia,gioia,neutrale,paura,rabbia,sorpresa,tristezza,All
orig,,,,,,,,,
aspettativa,473,0,2,0,2,1,12,0,490
fiducia,0,1,0,0,0,0,0,0,1
gioia,46,1,164,0,0,0,8,0,219
neutrale,0,0,0,2,0,0,0,0,2
paura,1,1,0,0,13,0,0,0,15
rabbia,1,0,0,0,0,2,0,0,3
sorpresa,0,0,0,0,0,0,6,0,6
tristezza,1,0,0,0,3,0,0,6,10
All,522,3,166,2,18,3,26,6,746


Questa matrice mostra come le emozioni dominanti si spostano tra l'originale e l'ibrido con α=0.5.

## Coverage: token non trovati in ELIta

In [10]:
# Analisi dei token del corpus non coperti da ELIta
_, _, unmatched = detect_emotions(df_corpus, df_tokens, df_elita_orig)
unmatched_counts = pd.Series(unmatched).value_counts()

print(f"Lemmi non in ELIta: {len(unmatched_counts)}")
print(f"\nTop 30 lemmi mancanti:")
print(unmatched_counts.head(30).to_string())

# Salviamo per riferimento nella relazione
unmatched_counts.to_csv(OUTPUT_DIR / "token_non_coperti.csv", header=["frequenza"])
print("\nSalvati in 'output_confronto/token_non_coperti.csv'")

Lemmi non in ELIta: 3968

Top 30 lemmi mancanti:
denota             1
topic              1
y.                 1
reality            1
dubbed             1
organolettico      1
scommetto          1
fastido            1
appalachia         1
facciale           1
conversare         1
intenzionalità     1
fatti              1
democratico        1
skippare           1
traduarr           1
testare la         1
ricordarsene       1
giovrira           1
dimentichi         1
shuffle            1
obiettare          1
libri              1
sensoriale         1
cantare la         1
mangae             1
penetrazione       1
erbetta            1
giocherellonare    1
infedelo           1

Salvati in 'output_confronto/token_non_coperti.csv'


## Salvataggio risultati completi

In [11]:
# Salviamo tutti i profili emotivi per ogni versione
for version_name, df_res in all_results.items():
    fname = "emotion_results_" + version_name.replace(" ", "_").replace("(", "").replace(")", "").replace("=", "") + ".csv"
    out_path = OUTPUT_DIR / fname
    df_res.to_csv(out_path, index=False)
    print(f"Salvato: {out_path}")

# Salviamo tutte le tabelle riassuntive in un unico file
df_all_tables = pd.concat(
    [tab.assign(Versione=vname) for vname, tab in all_tables.items()],
    ignore_index=True
)
df_all_tables.to_csv(OUTPUT_DIR / "emotion_tables_confronto.csv", index=False)
print("\nTutte le tabelle salvate in 'output_confronto/emotion_tables_confronto.csv'")
print(df_all_tables)

Salvato: output_confronto/emotion_results_Originale_α0.csv
Salvato: output_confronto/emotion_results_Ibrido_α0.2.csv
Salvato: output_confronto/emotion_results_Ibrido_α0.5.csv
Salvato: output_confronto/emotion_results_Ibrido_α0.8.csv

Tutte le tabelle salvate in 'output_confronto/emotion_tables_confronto.csv'
       Emozione  N. Commenti  N. Token         Versione
0         Gioia          744     18714  Originale (α=0)
1     Tristezza          744     18714  Originale (α=0)
2        Rabbia          744     18714  Originale (α=0)
3         Paura          744     18714  Originale (α=0)
4      Disgusto          744     18714  Originale (α=0)
5       Fiducia          744     18714  Originale (α=0)
6      Sorpresa          744     18714  Originale (α=0)
7   Aspettativa          744     18714  Originale (α=0)
8         Gioia          744     18714   Ibrido (α=0.2)
9     Tristezza          744     18714   Ibrido (α=0.2)
10       Rabbia          744     18714   Ibrido (α=0.2)
11        Paura   

---
## Problema: aspettativa e gioia sbilanciamo l'analisi

Come documentato in ItEm (Pollacci, 2015), queste due emozioni tendono a
produrre coseni sistematicamente più alti rispetto alle altre, diventando
"catalizzanti" e assorbendo falsi positivi. Lo stesso fenomeno è visibile
nei nostri dati.

Due soluzioni (entrambe da ItEm):
1. **corpus_6emo** — rimuoviamo aspettativa e gioia dall'analisi
2. **corpus_mean** — normalizziamo i punteggi usando la media per emozione

In [12]:
# Verifica del problema: quanto pesano aspettativa e gioia?
emo_dominant_counts = all_results["Originale (α=0)"]["dominant_emotion"].value_counts()

print("Distribuzione emozione dominante — Originale:")
print(emo_dominant_counts.to_string())
print()

total = emo_dominant_counts.sum()
for emo in ["aspettativa", "gioia"]:
    n = emo_dominant_counts.get(emo, 0)
    print(f"  {emo:15s}: {n:4d} commenti ({n/total*100:.1f}%)")

print()
positive = sum(emo_dominant_counts.get(e, 0) for e in ["gioia", "fiducia", "sorpresa", "aspettativa"])
negative = sum(emo_dominant_counts.get(e, 0) for e in ["tristezza", "rabbia", "paura", "disgusto"])
print(f"Totale positive : {positive} ({positive/total*100:.1f}%)")
print(f"Totale negative : {negative} ({negative/total*100:.1f}%)")

Distribuzione emozione dominante — Originale:
dominant_emotion
aspettativa    490
gioia          219
paura           15
tristezza       10
sorpresa         6
rabbia           3
neutrale         2
fiducia          1

  aspettativa    :  490 commenti (65.7%)
  gioia          :  219 commenti (29.4%)

Totale positive : 716 (96.0%)
Totale negative : 28 (3.8%)


### Soluzione 1 — corpus_6emo
Ripetiamo l'emotion detection escludendo aspettativa e gioia.
Utile per vedere la distribuzione delle emozioni "pure" senza i confounders.

In [13]:
SIX_EMOTIONS = ['fiducia', 'tristezza', 'rabbia', 'paura', 'disgusto', 'sorpresa']

SIX_COLORS = {k: v for k, v in EMOTION_COLORS.items() if k in SIX_EMOTIONS}

def detect_emotions_6(df_corpus, df_tokens, df_elita, pos_filter=None):
    """
    Emotion detection sulle sole 6 emozioni (esclude aspettativa e gioia).
    Identica alla detect_emotions ma ignora i punteggi delle due emozioni problematiche.
    """
    if pos_filter:
        df_filtered = df_tokens[df_tokens["pos"].isin(pos_filter)].copy()
    else:
        df_filtered = df_tokens.copy()

    elita_index = set(df_elita.index)
    tokens_by_comment = df_filtered.groupby("comment_id")["lemma"].apply(list).to_dict()

    results = []
    for _, row in df_corpus.iterrows():
        cid   = row["comment_id"]
        lemmi = tokens_by_comment.get(cid, [])

        emotion_scores = {emo: 0.0 for emo in SIX_EMOTIONS}
        found = []

        for lemma in lemmi:
            if lemma in elita_index:
                found.append(lemma)
                for emo in SIX_EMOTIONS:
                    emotion_scores[emo] += df_elita.loc[lemma, emo]

        results.append({
            "comment_id"      : cid,
            "n_tokens_matched": len(found),
            **emotion_scores,
            "dominant_emotion": max(emotion_scores, key=emotion_scores.get)
                                 if any(v > 0 for v in emotion_scores.values()) else "neutrale"
        })

    df_results = pd.DataFrame(results)

    table_rows = []
    for emo in SIX_EMOTIONS:
        n_comm = (df_results[emo] > 0).sum()
        n_tok  = df_filtered[
            df_filtered["comment_id"].isin(df_results[df_results[emo] > 0]["comment_id"]) &
            df_filtered["lemma"].isin(elita_index)
        ]["lemma"].count()
        table_rows.append({"Emozione": emo.capitalize(), "N. Commenti": int(n_comm), "N. Token": int(n_tok)})

    return df_results, pd.DataFrame(table_rows)


POS_FILTER = {"ADJ", "NOUN", "VERB"}

results_6emo = {}
for version_name, df_elita in MATRICES.items():
    df_res, df_tab = detect_emotions_6(df_corpus, df_tokens, df_elita, pos_filter=POS_FILTER)
    results_6emo[version_name] = df_res

print("corpus_6emo calcolato per tutte le versioni.")
print()

# Confronto distribuzione emozione dominante: 8 emo vs 6 emo (originale)
orig_8 = all_results["Originale (α=0)"]["dominant_emotion"].value_counts()
orig_6 = results_6emo["Originale (α=0)"]["dominant_emotion"].value_counts()

print("Confronto distribuzione — Originale:")
print(f"{'Emozione':15s} {'8 emo':>8s} {'6 emo':>8s}")
for emo in BASIC_EMOTIONS:
    n8 = orig_8.get(emo, 0)
    n6 = orig_6.get(emo, 0)
    print(f"{emo:15s} {n8:>8d} {n6:>8d}")

corpus_6emo calcolato per tutte le versioni.

Confronto distribuzione — Originale:
Emozione           8 emo    6 emo
gioia                219        0
tristezza             10       68
rabbia                 3       11
paura                 15       51
disgusto               0        2
fiducia                1      322
sorpresa               6      290
aspettativa          490        0


Si nota ora che sono fiducia e sorpresa ad essere predominanti, mentre aspettativa e gioia sono state rimosse. Le emozioni negative emergono più chiaramente. Ma questa soluzione è sbilanciata e non risolve il problema.

### Soluzione 2 — corpus_mean (normalizzazione per media)
Per ogni commento, invece di usare i punteggi assoluti, dividiamo ogni punteggio
emotivo per la somma totale degli score del commento. Questo azzera il vantaggio
sistematico di aspettativa e fiducia, esattamente come descritto in ItEm.

In [14]:
def normalize_by_mean(df_results):
    """
    Normalizzazione corpus_mean (da ItEm, Formula 3.5):
    Per ogni commento, ogni score emotivo viene diviso per la somma
    degli score di tutte le emozioni.
    Questo rimuove il vantaggio assoluto di aspettativa e gioia.
    """
    df_norm = df_results.copy()
    row_sums = df_results[BASIC_EMOTIONS].sum(axis=1).replace(0, 1)  # evita divisione per 0
    for emo in BASIC_EMOTIONS:
        df_norm[emo] = df_results[emo] / row_sums
    # Ricalcola emozione dominante con i punteggi normalizzati
    df_norm["dominant_emotion"] = df_norm[BASIC_EMOTIONS].idxmax(axis=1)
    df_norm.loc[df_results[BASIC_EMOTIONS].sum(axis=1) == 0, "dominant_emotion"] = "neutrale"
    return df_norm


results_mean = {}
for version_name, df_res in all_results.items():
    results_mean[version_name] = normalize_by_mean(df_res)

print("corpus_mean calcolato per tutte le versioni.")
print()

# Confronto distribuzione: originale vs mean
orig_raw  = all_results["Originale (α=0)"]["dominant_emotion"].value_counts()
orig_mean = results_mean["Originale (α=0)"]["dominant_emotion"].value_counts()
orig_6emo = results_6emo["Originale (α=0)"]["dominant_emotion"].value_counts()

print(f"{'Emozione':15s} {'Raw':>8s} {'Mean':>8s} {'6emo':>8s}")
for emo in BASIC_EMOTIONS:
    r = orig_raw.get(emo, 0)
    m = orig_mean.get(emo, 0)
    s = orig_6emo.get(emo, 0)
    print(f"{emo:15s} {r:>8d} {m:>8d} {s:>8d}")

corpus_mean calcolato per tutte le versioni.

Emozione             Raw     Mean     6emo
gioia                219      219        0
tristezza             10       10       68
rabbia                 3        3       11
paura                 15       15       51
disgusto               0        0        2
fiducia                1        1      322
sorpresa               6        6      290
aspettativa          490      490        0


### Visualizzazione comparativa: Raw vs Mean vs 6emo

In [15]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=["Raw (8 emozioni)", "Mean normalizzato (8 emo)", "6 emozioni (no asp/fid)"],
    horizontal_spacing=0.08
)

datasets = [
    all_results["Originale (α=0)"],
    results_mean["Originale (α=0)"],
    results_6emo["Originale (α=0)"],
]

emo_sets = [BASIC_EMOTIONS, BASIC_EMOTIONS, SIX_EMOTIONS]

for col_idx, (df_res, emo_set) in enumerate(zip(datasets, emo_sets), start=1):
    counts = df_res["dominant_emotion"].value_counts()
    total  = counts.sum()
    for emo in emo_set:
        n = counts.get(emo, 0)
        fig.add_trace(
            go.Bar(
                name=emo,
                x=[emo],
                y=[round(n / total * 100, 1)],
                marker_color=EMOTION_COLORS.get(emo, "#aaa"),
                showlegend=(col_idx == 1),
                legendgroup=emo,
                text=[f"{n/total*100:.0f}%"],
                textposition="outside"
            ),
            row=1, col=col_idx
        )

fig.update_layout(
    title_text="Impatto della normalizzazione sull'emozione dominante (ELIta Originale)",
    barmode="group",
    height=500,
    yaxis_title="% commenti",
    yaxis2_title="% commenti",
    yaxis3_title="% commenti"
)
fig.show()

### Bilancio positivo/negativo prima e dopo la normalizzazione

In [16]:
POSITIVE = {"gioia", "fiducia", "sorpresa", "aspettativa"}
NEGATIVE = {"tristezza", "rabbia", "paura", "disgusto"}

def balance_summary(df_res, label):
    counts = df_res["dominant_emotion"].value_counts()
    total  = counts.sum()
    pos = sum(counts.get(e, 0) for e in POSITIVE)
    neg = sum(counts.get(e, 0) for e in NEGATIVE)
    print(f"{label:35s} | Positive: {pos/total*100:5.1f}% | Negative: {neg/total*100:5.1f}%")

print(f"{'':35s} | {'Positive':>14s} | {'Negative':>14s}")
print("-" * 72)
for vname in MATRICES.keys():
    balance_summary(all_results[vname],    f"Raw     — {vname}")
    balance_summary(results_mean[vname],   f"Mean    — {vname}")
    balance_summary(results_6emo[vname],   f"6emo    — {vname}")
    print()

                                    |       Positive |       Negative
------------------------------------------------------------------------
Raw     — Originale (α=0)           | Positive:  96.0% | Negative:   3.8%
Mean    — Originale (α=0)           | Positive:  96.0% | Negative:   3.8%
6emo    — Originale (α=0)           | Positive:  82.0% | Negative:  17.7%

Raw     — Ibrido (α=0.2)            | Positive:  96.0% | Negative:   3.8%
Mean    — Ibrido (α=0.2)            | Positive:  96.0% | Negative:   3.8%
6emo    — Ibrido (α=0.2)            | Positive:  89.3% | Negative:  10.5%

Raw     — Ibrido (α=0.5)            | Positive:  96.1% | Negative:   3.6%
Mean    — Ibrido (α=0.5)            | Positive:  96.1% | Negative:   3.6%
6emo    — Ibrido (α=0.5)            | Positive:  93.8% | Negative:   5.9%

Raw     — Ibrido (α=0.8)            | Positive:  96.9% | Negative:   2.8%
Mean    — Ibrido (α=0.8)            | Positive:  96.9% | Negative:   2.8%
6emo    — Ibrido (α=0.8)            | Po

### Scelta del metodo consigliato
- **corpus_6emo**: più pulito per l'analisi esplorativa, utile nella relazione per mostrare le emozioni forti
- **corpus_mean**: mantiene tutte le 8 emozioni, fedele alla teoria di Plutchik — da usare come metodo principale
- I risultati del corpus_mean sono i più confrontabili con ELIta originale vs ricalcolato

In [17]:
# Salvataggio risultati normalizzati
for version_name in MATRICES.keys():
    fname_mean = OUTPUT_DIR / ("emotion_results_MEAN_" + version_name.replace(" ", "_").replace("(","").replace(")","").replace("=","") + ".csv")
    fname_6emo = OUTPUT_DIR / ("emotion_results_6EMO_" + version_name.replace(" ", "_").replace("(","").replace(")","").replace("=","") + ".csv")
    results_mean[version_name].to_csv(fname_mean, index=False)
    results_6emo[version_name].to_csv(fname_6emo, index=False)
    print(f"Salvati: {fname_mean}, {fname_6emo}")

Salvati: output_confronto/emotion_results_MEAN_Originale_α0.csv, output_confronto/emotion_results_6EMO_Originale_α0.csv
Salvati: output_confronto/emotion_results_MEAN_Ibrido_α0.2.csv, output_confronto/emotion_results_6EMO_Ibrido_α0.2.csv
Salvati: output_confronto/emotion_results_MEAN_Ibrido_α0.5.csv, output_confronto/emotion_results_6EMO_Ibrido_α0.5.csv
Salvati: output_confronto/emotion_results_MEAN_Ibrido_α0.8.csv, output_confronto/emotion_results_6EMO_Ibrido_α0.8.csv


---
## Analisi qualitativa del bias tematico

La diagnosi ha chiarito che il dominio di gioia/aspettativa non è un artefatto
matematico ma riflette il contenuto reale del corpus. Le celle seguenti lo
documentano in modo rigoroso per la relazione.

In [18]:
# Costruisce il sotto-lessico ELIta effettivamente presente nel corpus
# (serve per le sezioni "bias tematico" e "parole driver").
POS_FILTER = {"ADJ", "NOUN", "VERB"}

lemmi_corpus = (
    df_tokens[df_tokens["pos"].isin(POS_FILTER)]["lemma"]
    .dropna()
    .astype(str)
)

matched_in_elita = sorted(set(lemmi_corpus).intersection(set(df_elita_orig.index)))

df_corpus_elita = df_elita_orig.loc[matched_in_elita, BASIC_EMOTIONS].copy()
df_corpus_elita.index.name = "parola"

print(f"Lemmi del corpus presenti in ELIta: {len(df_corpus_elita)}")

# --- 1. Bias tematico: ELIta vs corpus generico ---
total_elita_words = len(df_elita_orig)
dominant_all = df_elita_orig[BASIC_EMOTIONS].idxmax(axis=1).value_counts()

print("Distribuzione emozione dominante in ELIta (intero lessico):")
for emo in BASIC_EMOTIONS:
    n = dominant_all.get(emo, 0)
    print(f"  {emo:15s}: {n:5d} ({n/total_elita_words*100:.1f}%)")

print()
print("Distribuzione emozione dominante nelle parole del corpus 'film':")
dominant_corpus = df_corpus_elita[BASIC_EMOTIONS].idxmax(axis=1).value_counts()
total_corpus_words = len(df_corpus_elita)
for emo in BASIC_EMOTIONS:
    n_corpus = dominant_corpus.get(emo, 0)
    n_elita  = dominant_all.get(emo, 0)
    diff = (n_corpus/total_corpus_words) - (n_elita/total_elita_words)
    arrow = "▲" if diff > 0.02 else ("▼" if diff < -0.02 else " ")
    print(f"  {emo:15s}: {n_corpus:5d} ({n_corpus/total_corpus_words*100:.1f}%)  {arrow} vs ELIta ({n_elita/total_elita_words*100:.1f}%)")

Lemmi del corpus presenti in ELIta: 2854
Distribuzione emozione dominante in ELIta (intero lessico):
  gioia          :  1562 (23.2%)
  tristezza      :   778 (11.6%)
  rabbia         :   704 (10.5%)
  paura          :   653 (9.7%)
  disgusto       :   248 (3.7%)
  fiducia        :   763 (11.4%)
  sorpresa       :   438 (6.5%)
  aspettativa    :  1573 (23.4%)

Distribuzione emozione dominante nelle parole del corpus 'film':
  gioia          :   700 (24.5%)    vs ELIta (23.2%)
  tristezza      :   288 (10.1%)    vs ELIta (11.6%)
  rabbia         :   243 (8.5%)    vs ELIta (10.5%)
  paura          :   257 (9.0%)    vs ELIta (9.7%)
  disgusto       :    64 (2.2%)    vs ELIta (3.7%)
  fiducia        :   348 (12.2%)    vs ELIta (11.4%)
  sorpresa       :   180 (6.3%)    vs ELIta (6.5%)
  aspettativa    :   774 (27.1%)  ▲ vs ELIta (23.4%)


In [19]:
# --- 2. Parole "driver" del bias: le più frequenti e più emotive ---
# Queste sono le parole che guidano il risultato finale. Documentarle è
# metodologicamente importante: dimostra che il risultato è riproducibile e motivato.

import plotly.express as px

# Per ogni parola matched, calcola: frequenza_nel_corpus × score_gioia (contributo totale)
freq_series = df_tokens[df_tokens["lemma"].isin(matched_in_elita)]["lemma"].value_counts()
freq_df = freq_series.reset_index()
freq_df.columns = ["lemma", "frequenza"]
freq_df = freq_df.merge(
    df_corpus_elita[BASIC_EMOTIONS].reset_index().rename(columns={"parola":"lemma"}),
    on="lemma"
)

# Emozione dominante per parola
freq_df["emo_dominante"] = freq_df[BASIC_EMOTIONS].idxmax(axis=1)
freq_df["score_dominante"] = freq_df[BASIC_EMOTIONS].max(axis=1)
# Contributo = frequenza × score emozione dominante
freq_df["contributo"] = freq_df["frequenza"] * freq_df["score_dominante"]

print("Top 20 parole per contributo emotivo totale al corpus:")
top_contrib = freq_df.sort_values("contributo", ascending=False).head(20)
print(top_contrib[["lemma","frequenza","emo_dominante","score_dominante","contributo"]].to_string(index=False))

fig = px.bar(
    top_contrib,
    x="lemma", y="contributo",
    color="emo_dominante",
    color_discrete_map=EMOTION_COLORS,
    title="Top 20 parole per contributo emotivo (frequenza × score)",
    labels={"contributo": "Contributo totale", "lemma": "Parola", "emo_dominante": "Emozione"}
)
fig.update_layout(xaxis_tickangle=-35)
fig.show()

Top 20 parole per contributo emotivo totale al corpus:
   lemma  frequenza emo_dominante  score_dominante  contributo
   avere       1024       fiducia             0.58      593.92
    film        944         gioia             0.58      547.52
    fare        558   aspettativa             0.58      323.64
  potere        295         gioia             0.67      197.65
  vedere        246         gioia             0.79      194.34
     più        307   aspettativa             0.42      128.94
   altro        246      sorpresa             0.50      123.00
    solo        204     tristezza             0.58      118.32
    anno        217   aspettativa             0.54      117.18
    bene        106         gioia             1.00      106.00
guardare        139   aspettativa             0.71       98.69
   stare        164   aspettativa             0.54       88.56
  volere        123   aspettativa             0.71       87.33
 piacere         81         gioia             0.96       77.76


In [20]:
# --- 3. Analisi per soglia di distintività ---
# Una parola è "distintiva" per un'emozione se il suo score è molto più alto
# delle altre. Filtriamo tenendo solo le parole dove:
# score_max - score_secondo > 0.3  (chiara dominanza su una sola emozione)
# Questo rimuove le parole "ambigue" che alzano più emozioni contemporaneamente.

def detect_emotions_distinctive(df_corpus, df_tokens, df_elita, distinctiveness_threshold=0.3):
    """
    Emotion detection con soglia di distintività.
    Usa solo le parole dove l'emozione dominante è chiaramente separata dalla seconda.
    """
    pos_filter = {"ADJ", "NOUN", "VERB"}
    df_filtered = df_tokens[df_tokens["pos"].isin(pos_filter)].copy()
    elita_index = set(df_elita.index)

    # Calcola distintività per ogni parola in ELIta
    sorted_scores = df_elita[BASIC_EMOTIONS].apply(lambda r: sorted(r.values, reverse=True), axis=1)
    df_elita_d = df_elita.copy()
    df_elita_d["distinctiveness"] = sorted_scores.apply(lambda s: s[0] - s[1] if len(s) > 1 else s[0])
    distinctive_words = set(df_elita_d[df_elita_d["distinctiveness"] >= distinctiveness_threshold].index)

    print(f"Parole con distintività >= {distinctiveness_threshold}: {len(distinctive_words)} su {len(df_elita)}")

    tokens_by_comment = df_filtered.groupby("comment_id")["lemma"].apply(list).to_dict()
    results = []

    for _, row in df_corpus.iterrows():
        cid   = row["comment_id"]
        lemmi = tokens_by_comment.get(cid, [])
        emotion_scores = {emo: 0.0 for emo in BASIC_EMOTIONS}
        found = 0

        for lemma in lemmi:
            if lemma in elita_index and lemma in distinctive_words:
                found += 1
                for emo in BASIC_EMOTIONS:
                    emotion_scores[emo] += df_elita.loc[lemma, emo]

        results.append({
            "comment_id"       : row["comment_id"],
            "n_tokens_matched" : found,
            **emotion_scores,
            "dominant_emotion" : max(emotion_scores, key=emotion_scores.get)
                                  if any(v > 0 for v in emotion_scores.values()) else "neutrale"
        })

    return pd.DataFrame(results)


# Testa con soglie diverse
for threshold in [0.2, 0.3, 0.5]:
    df_dist = detect_emotions_distinctive(df_corpus, df_tokens, df_elita_orig, threshold)
    counts  = df_dist["dominant_emotion"].value_counts()
    total   = counts.sum()
    neutri  = (df_dist["n_tokens_matched"] == 0).sum()
    print(f"\nSoglia {threshold} — commenti con match: {total - neutri} | neutri: {neutri}")
    for emo in BASIC_EMOTIONS:
        n = counts.get(emo, 0)
        print(f"  {emo:15s}: {n:4d} ({n/total*100:.1f}%)")

Parole con distintività >= 0.2: 1677 su 6719

Soglia 0.2 — commenti con match: 635 | neutri: 111
  gioia          :  181 (24.3%)
  tristezza      :   15 (2.0%)
  rabbia         :    5 (0.7%)
  paura          :   17 (2.3%)
  disgusto       :    2 (0.3%)
  fiducia        :   14 (1.9%)
  sorpresa       :    5 (0.7%)
  aspettativa    :  396 (53.1%)
Parole con distintività >= 0.3: 640 su 6719

Soglia 0.3 — commenti con match: 377 | neutri: 369
  gioia          :  106 (14.2%)
  tristezza      :   12 (1.6%)
  rabbia         :   10 (1.3%)
  paura          :   24 (3.2%)
  disgusto       :   14 (1.9%)
  fiducia        :   10 (1.3%)
  sorpresa       :   27 (3.6%)
  aspettativa    :  174 (23.3%)
Parole con distintività >= 0.5: 149 su 6719

Soglia 0.5 — commenti con match: 90 | neutri: 656
  gioia          :   43 (5.8%)
  tristezza      :   10 (1.3%)
  rabbia         :    7 (0.9%)
  paura          :    5 (0.7%)
  disgusto       :   15 (2.0%)
  fiducia        :    5 (0.7%)
  sorpresa       :    1 (0

In [21]:
# --- 4. Grafico finale: confronto Raw vs 6emo vs Distinctiveness 0.3 ---
df_dist_03 = detect_emotions_distinctive(df_corpus, df_tokens, df_elita_orig, 0.3)

fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=["Raw (tutte le parole)", "6 emozioni (no gioia/asp)", "Soglia distintività 0.3"],
    horizontal_spacing=0.08
)

datasets_final = [
    (all_results["Originale (α=0)"], BASIC_EMOTIONS),
    (results_6emo["Originale (α=0)"], SIX_EMOTIONS),
    (df_dist_03, BASIC_EMOTIONS),
]

for col_idx, (df_res, emo_set) in enumerate(datasets_final, start=1):
    counts = df_res["dominant_emotion"].value_counts()
    total  = counts.sum()
    for emo in emo_set:
        n = counts.get(emo, 0)
        fig.add_trace(
            go.Bar(
                name=emo,
                x=[emo], y=[round(n/total*100, 1)],
                marker_color=EMOTION_COLORS.get(emo, "#aaa"),
                showlegend=(col_idx == 1),
                legendgroup=emo,
                text=[f"{n/total*100:.0f}%"],
                textposition="outside"
            ),
            row=1, col=col_idx
        )

fig.update_layout(
    title_text="Strategie di analisi a confronto (ELIta Originale)",
    barmode="group", height=520
)
fig.show()

# Salva anche la versione con soglia distintività
df_dist_03.to_csv(OUTPUT_DIR / "emotion_results_DISTINCTIVE_03_Originale.csv", index=False)
print("Salvato output_confronto/emotion_results_DISTINCTIVE_03_Originale.csv")

Parole con distintività >= 0.3: 640 su 6719


Salvato output_confronto/emotion_results_DISTINCTIVE_03_Originale.csv


---
## Conclusione metodologica

I risultati di questa analisi sono coerenti e motivabili:

**Raw**: gioia e aspettativa dominano perché il topic "estate" è semanticamente
positivo. Le parole più frequenti del corpus (*iniziare*, *mare*, *natale*, *figlio*,
*crescere*) hanno score 1.0 su gioia/aspettativa in ELIta — non è un artefatto.

**corpus_6emo**: rimuovendo gioia e aspettativa emerge la struttura emotiva
secondaria del corpus. Fiducia e sorpresa diventano dominanti, con una presenza
più visibile di emozioni negative (paura, tristezza). Utile per l'analisi interna.

**Soglia distintività**: l'approccio più rigoroso. Usa solo parole con un segnale
emotivo netto. Riduce il rumore delle parole "ambigue" (alta gioia *e* alta aspettativa)
e mostra una distribuzione più bilanciata.

**Per la relazione**: presenta tutti e tre i risultati. Il confronto Raw vs Distintività
è la dimostrazione più chiara dell'impatto del metodo di ponderazione — ed è
direttamente collegato al confronto ELIta originale vs ricalcolato.

---
## Problema reale: verbi e nomi generici dominano il risultato

La cella di diagnostica mostra che le parole con più contributo emotivo sono
`avere`, `fare`, `potere`, `vedere`, `stare`, `volere` — verbi generici
frequentissimi in qualsiasi testo italiano, non portatori di emozione specifica.
La soluzione è una **lista di esclusione** (emotional stopwords):
parole presenti in ELIta ma semanticamente vuote nel contesto dell'analisi emotiva.

In [22]:
# Verbi modali, ausiliari e generici da escludere
# Sono presenti in ELIta perché la matrice include tutte le parole,
# ma non portano contenuto emotivo in un testo qualsiasi.
EMOTIONAL_STOPWORDS = {
    # Verbi ausiliari / modali / supporto
    "avere", "essere", "fare", "stare", "dare", "andare", "venire",
    "potere", "volere", "dovere", "sapere", "vedere", "sentire",
    "trovare", "pensare", "dire", "parlare", "guardare", "tenere",
    "portare", "prendere", "mettere", "lasciare", "passare", "uscire",
    "entrare", "tornare", "rimanere", "iniziare", "finire", "continuare",
    # Nomi generici / funzionali
    "cosa", "modo", "parte", "punto", "volta", "anno", "tempo", "caso",
    "fatto", "posto", "tipo", "gente", "persona", "vita", "mio", "tuo",
    # Aggettivi generici
    "altro", "solo", "grande", "piccolo", "nuovo", "vecchio", "primo",
    "ultimo", "stesso", "proprio", "bello", "buono",
    # Avverbi (se finiscono in ELIta)
    "più", "bene", "male", "molto", "poco", "tanto",
}

print(f"Emotional stopwords definite: {len(EMOTIONAL_STOPWORDS)}")

# Quante di queste sono effettivamente in ELIta?
in_elita = [w for w in EMOTIONAL_STOPWORDS if w in df_elita_orig.index]
print(f"Presenti in ELIta: {len(in_elita)}")
print()

# Quanto peso tolgono?
freq_stop = df_tokens[df_tokens["lemma"].isin(in_elita)]["lemma"].value_counts()
freq_total = len(df_tokens[df_tokens["pos"].isin({"ADJ","NOUN","VERB"})])
freq_stop_total = freq_stop.sum()
print(f"Token rimossi: {freq_stop_total} su {freq_total} ({freq_stop_total/freq_total*100:.1f}%)")
print()
print("Top 15 stopwords per frequenza nel corpus:")
print(freq_stop.head(15).to_string())

Emotional stopwords definite: 65
Presenti in ELIta: 61

Token rimossi: 7054 su 25028 (28.2%)

Top 15 stopwords per frequenza nel corpus:
lemma
avere       1024
fare         558
più          307
potere       295
altro        246
vedere       246
cosa         228
anno         217
solo         204
mio          179
dire         164
stare        164
dovere       152
guardare     139
persona      127


In [23]:
def detect_emotions_clean(df_corpus, df_tokens, df_elita,
                          stopwords=None, distinctiveness_threshold=0.0):
    """
    Emotion detection con filtro stopwords emotive e soglia di distintività.
    Parametri combinabili:
      stopwords               : set di lemmi da escludere
      distinctiveness_threshold: soglia minima score_max - score_secondo (0 = nessuna)
    """
    pos_filter = {"ADJ", "NOUN", "VERB"}
    df_filtered = df_tokens[df_tokens["pos"].isin(pos_filter)].copy()

    # Rimuovi stopwords
    if stopwords:
        df_filtered = df_filtered[~df_filtered["lemma"].isin(stopwords)]

    elita_index = set(df_elita.index)

    # Calcola distintività se richiesto
    if distinctiveness_threshold > 0:
        sorted_scores = df_elita[BASIC_EMOTIONS].apply(
            lambda r: sorted(r.values, reverse=True), axis=1)
        distinctiveness = sorted_scores.apply(lambda s: s[0] - s[1] if len(s) > 1 else s[0])
        valid_words = set(df_elita[distinctiveness >= distinctiveness_threshold].index)
        elita_index = elita_index & valid_words

    tokens_by_comment = df_filtered.groupby("comment_id")["lemma"].apply(list).to_dict()
    results = []

    for _, row in df_corpus.iterrows():
        cid   = row["comment_id"]
        lemmi = tokens_by_comment.get(cid, [])
        emotion_scores = {emo: 0.0 for emo in BASIC_EMOTIONS}
        found = 0

        for lemma in lemmi:
            if lemma in elita_index:
                found += 1
                for emo in BASIC_EMOTIONS:
                    emotion_scores[emo] += df_elita.loc[lemma, emo]

        results.append({
            "comment_id"       : row["comment_id"],
            "n_tokens_matched" : found,
            **emotion_scores,
            "dominant_emotion" : max(emotion_scores, key=emotion_scores.get)
                                  if found > 0 else "neutrale"
        })

    return pd.DataFrame(results)


# Testa le combinazioni
configs = [
    ("Raw",                         None,                 0.0),
    ("No stopwords",                EMOTIONAL_STOPWORDS,  0.0),
    ("No stopwords + distict. 0.2", EMOTIONAL_STOPWORDS,  0.2),
    ("No stopwords + distict. 0.3", EMOTIONAL_STOPWORDS,  0.3),
]

print(f"{'Configurazione':35s} {'gioia':>7s} {'asp':>7s} {'trist':>7s} {'rabbia':>7s} {'paura':>7s} {'neutri':>7s}")
print("-" * 85)
for label, sw, thr in configs:
    df_r = detect_emotions_clean(df_corpus, df_tokens, df_elita_orig, sw, thr)
    c    = df_r["dominant_emotion"].value_counts()
    tot  = len(df_r)
    print(f"{label:35s} {c.get('gioia',0):>7d} {c.get('aspettativa',0):>7d} "
          f"{c.get('tristezza',0):>7d} {c.get('rabbia',0):>7d} "
          f"{c.get('paura',0):>7d} {c.get('neutrale',0):>7d}")

Configurazione                        gioia     asp   trist  rabbia   paura  neutri
-------------------------------------------------------------------------------------
Raw                                     219     490      10       3      15       2
No stopwords                            245     431      14      13      27       2
No stopwords + distict. 0.2             161     286      25      13      29     184
No stopwords + distict. 0.3             108     168      12      11      25     379


In [24]:
# Grafico finale: Raw vs No-stopwords vs No-stopwords+Dist0.2
df_clean     = detect_emotions_clean(df_corpus, df_tokens, df_elita_orig, EMOTIONAL_STOPWORDS, 0.0)
df_clean_d02 = detect_emotions_clean(df_corpus, df_tokens, df_elita_orig, EMOTIONAL_STOPWORDS, 0.2)

fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=["Raw", "No stopwords generiche", "No stopwords + Dist. ≥ 0.2"],
    horizontal_spacing=0.08
)

for col_idx, df_r in enumerate([all_results["Originale (α=0)"], df_clean, df_clean_d02], start=1):
    counts = df_r["dominant_emotion"].value_counts()
    total  = len(df_r)
    for emo in BASIC_EMOTIONS + ["neutrale"]:
        n = counts.get(emo, 0)
        fig.add_trace(
            go.Bar(
                name=emo,
                x=[emo], y=[round(n/total*100, 1)],
                marker_color=EMOTION_COLORS.get(emo, "#999"),
                showlegend=(col_idx == 1),
                legendgroup=emo,
                text=[f"{n/total*100:.0f}%"],
                textposition="outside"
            ),
            row=1, col=col_idx
        )

fig.update_layout(
    title_text="Impatto del filtro stopwords emotive (corpus 'film')",
    barmode="group", height=520
)
fig.show()

# Salva la versione pulita
df_clean.to_csv("emotion_results_CLEAN_Originale.csv", index=False)
df_clean_d02.to_csv("emotion_results_CLEAN_D02_Originale.csv", index=False)
print("Salvati emotion_results_CLEAN e CLEAN_D02.")

Salvati emotion_results_CLEAN e CLEAN_D02.


---
## Confronto ELIta originale vs ricalcolato: cosa cambia davvero?

Il confronto tra le 4 versioni di ELIta finora mostrava sempre gli stessi numeri
di emozione dominante. Questo accade perché l'emozione dominante è una variabile
discreta (winner-takes-all) e piccole variazioni negli score non la cambiano.

Per confrontare le versioni in modo significativo bisogna guardare:
1. **I punteggi assoluti medi** per emozione — cambiano anche se la dominante no
2. **I commenti che cambiano emozione dominante** tra versioni (già fatto, 10.6%)
3. **La distanza tra primo e secondo classificato** — il ricalcolo la aumenta?
4. **La separazione degli cluster** — le emozioni diventano più distinte con α alto?

In [25]:
# 1. Punteggi medi per emozione nelle 4 versioni (con filtro stopwords)
print("Punteggi emotivi medi per versione (filtro stopwords attivo):")
print(f"{'Emozione':15s}", end="")
for vname in MATRICES.keys():
    print(f" {vname[:12]:>13s}", end="")
print()
print("-" * 75)

# Calcola risultati puliti per tutte le versioni
results_clean = {}
for vname, df_elita in MATRICES.items():
    results_clean[vname] = detect_emotions_clean(df_corpus, df_tokens, df_elita, EMOTIONAL_STOPWORDS, 0.0)

for emo in BASIC_EMOTIONS:
    print(f"{emo:15s}", end="")
    for vname in MATRICES.keys():
        mean_score = results_clean[vname][emo].mean()
        print(f" {mean_score:>13.4f}", end="")
    print()

Punteggi emotivi medi per versione (filtro stopwords attivo):
Emozione         Originale (α  Ibrido (α=0.  Ibrido (α=0.  Ibrido (α=0.
---------------------------------------------------------------------------
gioia                  7.3398        8.2511        9.6181       10.9851
tristezza              4.2537        5.1273        6.4376        7.7480
rabbia                 3.7636        4.7043        6.1153        7.5264
paura                  4.4544        5.4878        7.0380        8.5882
disgusto               2.2655        3.1399        4.4515        5.7630
fiducia                5.7527        7.0915        9.0996       11.1077
sorpresa               5.6315        7.0922        9.2832       11.4741
aspettativa            8.2185        9.2399       10.7720       12.3042


In [26]:
# 2. Distanza tra emozione dominante e seconda classificata
# Misura quanto è "sicura" l'assegnazione emotiva.
# Se il ricalcolo aumenta questa distanza, significa che le classi diventano più nette.

print("Distanza media tra score 1° e 2° classificato per versione:")
print(f"{'Versione':25s} {'Distanza media':>15s} {'Distanza mediana':>17s}")
print("-" * 60)

for vname in MATRICES.keys():
    df_r = results_clean[vname]
    scores = df_r[BASIC_EMOTIONS]
    # Ordina ogni riga e calcola differenza tra max e secondo
    sorted_scores = scores.apply(lambda r: sorted(r.values, reverse=True), axis=1)
    gap = sorted_scores.apply(lambda s: s[0] - s[1])
    # Escludi commenti senza match (gap = 0)
    gap_nonzero = gap[gap > 0]
    print(f"{vname:25s} {gap_nonzero.mean():>15.4f} {gap_nonzero.median():>17.4f}")

Distanza media tra score 1° e 2° classificato per versione:
Versione                   Distanza media  Distanza mediana
------------------------------------------------------------
Originale (α=0)                    1.1435            0.5300
Ibrido (α=0.2)                     1.1181            0.4962
Ibrido (α=0.5)                     1.0039            0.4156
Ibrido (α=0.8)                     0.7346            0.3313


In [27]:
# 3. Variazione degli score per emozione: originale vs α=0.8
# Mostra parola per parola quanto cambia il punteggio emotivo con il ricalcolo.
# Questo è l'effetto diretto della formula e* = α·cos + (1-α)·e

df_orig = df_elita_orig
df_08   = MATRICES["Ibrido (α=0.8)"]

# Solo parole presenti in entrambe le matrici
common_words = df_orig.index.intersection(df_08.index)
df_delta = (df_08.loc[common_words] - df_orig.loc[common_words]).abs()

print("Variazione media degli score per emozione (|α=0.8 - originale|):")
print(df_delta.mean().sort_values(ascending=False).round(4).to_string())
print()
print("Parole che cambiano di più (top 10 per variazione totale):")
df_delta["variazione_totale"] = df_delta[BASIC_EMOTIONS].sum(axis=1)
top_changed = df_delta.sort_values("variazione_totale", ascending=False).head(10)
# Mostra anche emozione originale e nuova
for word in top_changed.index:
    orig_emo = df_orig.loc[word, BASIC_EMOTIONS].idxmax()
    new_emo  = df_08.loc[word, BASIC_EMOTIONS].idxmax()
    arrow = f"{orig_emo} → {new_emo}" if orig_emo != new_emo else f"{orig_emo} (invariata)"
    print(f"  {word:20s} Δtotale={top_changed.loc[word,'variazione_totale']:.3f}  {arrow}")

Variazione media degli score per emozione (|α=0.8 - originale|):
sorpresa       0.3121
fiducia        0.2740
aspettativa    0.2348
paura          0.2266
gioia          0.2141
rabbia         0.2090
tristezza      0.2009
disgusto       0.1923

Parole che cambiano di più (top 10 per variazione totale):
  dito                 Δtotale=4.153  gioia → sorpresa
  editoriale           Δtotale=4.153  gioia → sorpresa
  fetta                Δtotale=4.038  aspettativa (invariata)
  provinciale          Δtotale=3.881  fiducia (invariata)
  elenco               Δtotale=3.814  tristezza → sorpresa
  fabbro               Δtotale=3.808  gioia → fiducia
  plastica             Δtotale=3.803  rabbia (invariata)
  fermare              Δtotale=3.786  paura (invariata)
  bionda               Δtotale=3.720  gioia (invariata)
  rugosità             Δtotale=3.686  disgusto (invariata)


In [28]:
# 4. Grafico riassuntivo: punteggi medi per emozione nelle 4 versioni
import plotly.express as px

means_data = []
for vname in MATRICES.keys():
    for emo in BASIC_EMOTIONS:
        means_data.append({
            "Versione": vname,
            "Emozione": emo.capitalize(),
            "Score medio": results_clean[vname][emo].mean()
        })

df_means_plot = pd.DataFrame(means_data)

fig = px.bar(
    df_means_plot,
    x="Emozione", y="Score medio",
    color="Versione",
    barmode="group",
    title="Score emotivo medio per versione ELIta (corpus 'film', no stopwords generiche)",
    color_discrete_sequence=["#455A64", "#1E88E5", "#FB8C00", "#E53935"],
    text_auto=".3f"
)
fig.update_traces(textposition="outside", textfont_size=9)
fig.update_layout(height=500)
fig.show()

print()
print("Nota: se le barre delle 4 versioni sono quasi identiche, significa che")
print("il ricalcolo cambia poco i punteggi assoluti medi ma può cambiare")
print("la distribuzione interna di singoli commenti (vedi matrice di transizione).")


Nota: se le barre delle 4 versioni sono quasi identiche, significa che
il ricalcolo cambia poco i punteggi assoluti medi ma può cambiare
la distribuzione interna di singoli commenti (vedi matrice di transizione).
